# 코스피200 야간선물 ↔ 반도체 종목 시초가 백테스트

## 목적
반도체 섹터 대표 종목 15개 각각의 정규장 시초가가 **코스피200 야간선물**과 명확한 상관관계가 있는지 확인한다.

- **분석 1. 상관관계**: 야간선물 수익률과 종목 시가갭의 상관계수·회귀계수가 통계적으로 유의한가
- **분석 2. 예측력**: 야간선물로 시초가를 예측하면 "전일 종가 그대로"보다 오차가 줄어드는가 (2026년 8월 walk-forward)

## 시간 순서

```
        t-1일 정규장               KRX 야간세션               t일 정규장
     09:00 ──── 15:45       18:00 ──────────── 06:00          09:00
                  ▲                              ▲               ▲
          F_close[t-1]                     F_night[t]      P_open[i,t]   ← 예측 대상
```

## 기호

| 기호 | 의미 | 컬럼 |
|---|---|---|
| $P^{open}_{i,t}$ | 종목 $i$의 $t$일 시가 | `open` |
| $P^{base}_{i,t}$ | 기준가 = 종가 − 전일대비 (보통 전일 종가, 액면분할·권리락 시 조정값) | `base` |
| $F^{night}_t$ | $t$일 개장 직전(06:00) 코스피200 야간선물 종가 | `night_close` |
| $F^{close}_{t-1}$ | 전일 코스피200 선물 정규장 종가 | `fut_close_prev` |
| $\text{night\_ret}_t$ | 야간선물 수익률 $= F^{night}_t / F^{close}_{t-1} - 1$ | `night_ret` |

$$\text{gap}_{i,t} = \frac{P^{open}_{i,t}}{P^{base}_{i,t}} - 1 \qquad\qquad \text{night\_ret}_t = \frac{F^{night}_t}{F^{close}_{t-1}} - 1$$

## 데이터
KRX Open API (D11·D12와 동일한 `AUTH_KEY` 헤더 방식, `.env`의 `KRX_API_KEY` 사용). **API별 이용신청 필요**

| API | 용도 |
|---|---|
| `drv/fut_bydd_trd` | 선물 일별매매정보 (코스피200 선물 정규·야간) |
| `sto/stk_bydd_trd` | 유가증권 일별매매정보 |
| `sto/ksq_bydd_trd` | 코스닥 일별매매정보 |

In [ ]:
import os
import time
from getpass import getpass
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from dotenv import find_dotenv, load_dotenv
from IPython.display import display
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

## 1. 설정

In [ ]:
# 반도체 섹터 대표 종목 (코스피·코스닥 모두 가능, 시장은 데이터에서 자동 판별)
STOCKS = {
    # 종목코드: (종목명, 분야)
    "005930": ("삼성전자", "메모리·파운드리"),
    "000660": ("SK하이닉스", "메모리"),
    "000990": ("DB하이텍", "파운드리"),
    "042700": ("한미반도체", "후공정 장비"),
    "039030": ("이오테크닉스", "레이저 장비"),
    "240810": ("원익IPS", "전공정 장비"),
    "036930": ("주성엔지니어링", "전공정 장비"),
    "403870": ("HPSP", "전공정 장비"),
    "089030": ("테크윙", "테스트 장비"),
    "058470": ("리노공업", "테스트 부품"),
    "095340": ("ISC", "테스트 부품"),
    "357780": ("솔브레인", "소재"),
    "005290": ("동진쎄미켐", "소재"),
    "064760": ("티씨케이", "소모품"),
    "067310": ("하나마이크론", "패키징·테스트"),
}

BACKTEST_START = "2026-08-01"
BACKTEST_END = "2026-08-31"

# 회귀 학습에 쓰는 직전 거래일 수 (rolling window)
# t-1부터 거꾸로 "데이터가 있는 거래일"을 센다 → 학습 표본은 항상 TRAIN_WINDOW개
TRAIN_WINDOW = 60
MIN_TRAIN = 20      # 학습 표본이 이보다 적은 날은 예측하지 않음

# 과거 데이터 수집 시작일 (달력 150일 ≈ 100거래일). 분석 1(상관관계)은 이 기간 전체를 쓴다.
DATA_START = pd.Timestamp(BACKTEST_START) - pd.Timedelta(days=150)

SIGNIFICANCE = 0.05

# 야간세션 BAS_DD 표기 방식
#   "same" : t일 직전 야간세션의 날짜 = t   (거래일 기준)
#   "prev" : t일 직전 야간세션의 날짜 = t-1 (야간장이 시작한 날)
#   "auto" : 두 방식 중 야간종가와 다음 정규장 선물시가가 더 가까운 쪽을 자동 선택
NIGHT_ALIGN = "auto"

FUTURES_PRODUCT = "코스피200선물"   # 공백 제거 후 비교

# 노트북이 있는 폴더(backtest/) 기준
BASE_DIR = Path.cwd()
CACHE_DIR = BASE_DIR / "data" / "krx_cache"
OUTPUT_DIR = BASE_DIR / "outputs"

KRX_BASE_URL = "https://data-dbg.krx.co.kr/svc/apis/"
ENDPOINTS = {
    "futures": "drv/fut_bydd_trd",
    "kospi": "sto/stk_bydd_trd",
    "kosdaq": "sto/ksq_bydd_trd",
}

## 2. KRX 데이터 수집

- KRX Open API는 **날짜 하나당 한 번** 호출하면 그날의 전 종목(선물은 전 상품)을 돌려준다.
- 받은 결과는 `data/krx_cache/{API}/{날짜}.csv`에 저장하고, 다음 실행부터는 다시 호출하지 않는다.
- 휴장일은 빈 파일로 저장한다. 단, 최근 7일 이내는 아직 데이터가 안 올라왔을 수 있어 빈 결과를 저장하지 않는다.

In [ ]:
def get_auth_key():
    load_dotenv(find_dotenv(usecwd=True))
    return os.getenv("KRX_API_KEY") or getpass("KRX API KEY 입력: ")


def fetch_krx_daily(auth_key, name, base_date):
    """KRX Open API 하루치 조회. 휴장일은 빈 DataFrame (빈 파일로 캐시)."""
    cache_path = CACHE_DIR / name / f"{base_date}.csv"

    if cache_path.exists():
        if cache_path.stat().st_size == 0:
            return pd.DataFrame()
        return pd.read_csv(cache_path, dtype=str)

    response = requests.get(
        KRX_BASE_URL + ENDPOINTS[name],
        headers={"AUTH_KEY": auth_key},
        params={"basDd": base_date},
        timeout=60,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"[{name} {base_date}] HTTP {response.status_code}: {response.text[:500]}"
        )

    data = response.json()

    if "OutBlock_1" not in data:
        raise RuntimeError(
            f"[{name} {base_date}] 정상적인 API 응답 구조가 아닙니다: {str(data)[:500]}"
        )

    df = pd.DataFrame(data["OutBlock_1"])

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    if df.empty:
        # 최근 날짜는 아직 데이터가 안 올라왔을 수 있으므로 빈 결과(휴장일)로 캐시하지 않는다
        if pd.Timestamp(base_date) < pd.Timestamp.today().normalize() - pd.Timedelta(days=7):
            cache_path.write_text("")
    else:
        df.to_csv(cache_path, index=False)

    time.sleep(0.2)
    return df


def fetch_range(auth_key, name, start, end):
    frames = [
        fetch_krx_daily(auth_key, name, day.strftime("%Y%m%d"))
        for day in pd.bdate_range(start, end)
    ]
    frames = [df for df in frames if not df.empty]

    if not frames:
        raise RuntimeError(f"[{name}] {start} ~ {end} 조회 결과가 없습니다.")

    return pd.concat(frames, ignore_index=True)

In [ ]:
auth_key = get_auth_key()

print(f"KRX 데이터 수집: {DATA_START:%Y-%m-%d} ~ {BACKTEST_END}")
fut_raw = fetch_range(auth_key, "futures", DATA_START, BACKTEST_END)
kospi_raw = fetch_range(auth_key, "kospi", DATA_START, BACKTEST_END)
kosdaq_raw = fetch_range(auth_key, "kosdaq", DATA_START, BACKTEST_END)

print(f"선물 {len(fut_raw):,}행 | 유가증권 {len(kospi_raw):,}행 | 코스닥 {len(kosdaq_raw):,}행")

## 3. 전처리

**선물**: 코스피200 선물(미니·스프레드 제외)만 남기고, `MKT_NM`으로 정규(day) / 야간(night) 세션을 구분한다.

**종목**: `STOCKS` 종목만 남기고 시가와 기준가를 정리한다.

$$P^{base}_{i,t} = \text{종가}_{i,t} - \text{전일대비}_{i,t}$$

KRX 일별 데이터는 수정주가가 아닌 원주가다. 기준가는 보통 전일 종가와 같고, 액면분할·권리락이 있는 날에는 조정된 값이 된다.
그래서 전일 종가 대신 기준가로 시가갭을 계산하면 가짜 갭이 생기지 않는다. 거래정지일(시가 0)은 제외한다.

In [ ]:
def _norm(s):
    """공백 제거 ("코스피 200" → "코스피200")"""
    return s.astype(str).str.replace(r"\s+", "", regex=True)


def _to_num(s):
    """"1,234.56" / "-" → 1234.56 / NaN"""
    return pd.to_numeric(s.astype(str).str.replace(",", "", regex=False), errors="coerce")


def prepare_futures(raw):
    """코스피200 선물(스프레드 제외)만 남기고 정규(day)/야간(night) 세션을 구분한다."""
    product = _norm(raw["PROD_NM"])
    name = _norm(raw["ISU_NM"])

    df = raw[(product == FUTURES_PRODUCT) & ~name.str.contains("SP|스프레드")].copy()

    if df.empty:
        raise ValueError(
            f"PROD_NM이 '{FUTURES_PRODUCT}'인 행이 없습니다. 실제 값: {sorted(product.unique())}"
        )

    market = _norm(df["MKT_NM"])

    if not market.str.contains("야간").any():
        raise ValueError(f"야간 세션 행이 없습니다. MKT_NM 실제 값: {sorted(market.unique())}")

    return pd.DataFrame({
        "date": pd.to_datetime(df["BAS_DD"], format="%Y%m%d"),
        "session": np.where(market.str.contains("야간"), "night", "day"),
        "contract": df["ISU_CD"],
        "contract_name": df["ISU_NM"],
        "open": _to_num(df["TDD_OPNPRC"]),
        "close": _to_num(df["TDD_CLSPRC"]),
        "volume": _to_num(df["ACC_TRDVOL"]),
    }).drop_duplicates(["date", "session", "contract"])


def prepare_stocks(raw):
    """STOCKS 종목만 남기고 날짜별 시가·기준가를 정리한다. 거래정지일(시가 0)은 제외."""
    df = raw[raw["ISU_CD"].isin(STOCKS)]

    missing = [f"{code} {STOCKS[code][0]}" for code in STOCKS if code not in set(df["ISU_CD"])]
    if missing:
        print(f"[경고] 데이터에 없는 종목 (상장폐지·코드 변경 확인 필요): {', '.join(missing)}")

    if df.empty:
        raise ValueError("STOCKS의 종목이 데이터에 하나도 없습니다.")

    close = _to_num(df["TDD_CLSPRC"])

    stocks = pd.DataFrame({
        "date": pd.to_datetime(df["BAS_DD"], format="%Y%m%d"),
        "code": df["ISU_CD"],
        "name": df["ISU_CD"].map(lambda code: STOCKS[code][0]),
        "market": df["MKT_NM"],
        "open": _to_num(df["TDD_OPNPRC"]),                  # P_open[i,t]
        "base": close - _to_num(df["CMPPREVDD_PRC"]),       # P_base[i,t] = 종가 - 전일대비
    })

    return (
        stocks[stocks["open"] > 0]
        .dropna()
        .drop_duplicates(["date", "code"])
        .sort_values(["code", "date"])
        .reset_index(drop=True)
    )

In [ ]:
fut = prepare_futures(fut_raw)
stocks = prepare_stocks(pd.concat([kospi_raw, kosdaq_raw], ignore_index=True))

# 정규장 거래일 = 유가증권시장 데이터가 있는 날
trading_days = pd.DatetimeIndex(
    sorted(pd.to_datetime(kospi_raw["BAS_DD"], format="%Y%m%d").unique())
)

print("거래일:", len(trading_days), "| 종목:", stocks["code"].nunique())
display(fut[fut["session"] == "night"].head())
display(stocks.head())

### 야간세션 매칭

정규장 거래일 $t$마다 **$t$일 개장 직전에 끝난 야간세션**을 붙인다.

- **월물**: 그 야간세션에서 거래량이 가장 많은 종목(= 최근월물). 전일 종가·당일 시가도 같은 종목에서 가져와 만기 교체 때 가격이 튀지 않게 한다.
- **날짜 표기 판별**: KRX가 야간세션에 다음 거래일 날짜를 붙이는지(`same`), 시작한 날 날짜를 붙이는지(`prev`) 확실하지 않아 자동 판별한다.
  올바른 방식이면 야간종가와 다음 정규장 선물시가가 3시간 차이라 거의 같고, 틀린 방식이면 하루 차이라 크게 벌어진다.

$$\text{night\_ret}_t = \frac{F^{night}_t}{F^{close}_{t-1}} - 1$$

In [ ]:
def build_night_dataset(fut, trading_days, align):
    """정규장 거래일 t마다 한 행:  F_close[t-1],  F_night[t],  night_ret[t]"""
    day_session = fut[fut["session"] == "day"].set_index(["date", "contract"])
    fut_open = day_session["open"].to_dict()     # (날짜, 종목) → 정규장 시가
    fut_close = day_session["close"].to_dict()   # (날짜, 종목) → 정규장 종가

    night_session = fut[
        (fut["session"] == "night") & fut["close"].notna() & (fut["volume"] > 0)
    ]
    night_by_date = dict(tuple(night_session.groupby("date")))

    rows = []

    for t_prev, t in zip(trading_days[:-1], trading_days[1:]):
        night_date = t if align == "same" else t_prev

        if night_date not in night_by_date:
            continue

        night = night_by_date[night_date]
        front = night.loc[night["volume"].idxmax()]   # 최근월물
        contract = front["contract"]

        rows.append({
            "date": t,
            "contract_name": front["contract_name"],
            "night_close": front["close"],                               # F_night[t]
            "fut_close_prev": fut_close.get((t_prev, contract), np.nan), # F_close[t-1]
            "fut_open": fut_open.get((t, contract), np.nan),             # 선물 정규장 시가[t]
        })

    if not rows:
        return pd.DataFrame()

    night = (
        pd.DataFrame(rows)
        .set_index("date")
        .dropna(subset=["night_close", "fut_close_prev"])
    )

    # night_ret[t] = F_night[t] / F_close[t-1] - 1
    night["night_ret"] = night["night_close"] / night["fut_close_prev"] - 1

    return night


def resolve_alignment(fut, trading_days):
    """야간세션 날짜 표기 방식 판별 (설명은 위 마크다운 참고)"""
    if NIGHT_ALIGN != "auto":
        return NIGHT_ALIGN, build_night_dataset(fut, trading_days, NIGHT_ALIGN)

    print("야간세션 날짜 표기 방식 자동 판별")

    datasets, gaps = {}, {}

    for align in ("same", "prev"):
        night = build_night_dataset(fut, trading_days, align)
        datasets[align] = night
        gaps[align] = (
            (night["fut_open"] / night["night_close"] - 1).abs().median()
            if not night.empty else np.inf
        )
        print(
            f"  {align:>4}: 표본 {len(night)}일, "
            f"|정규장 선물시가 / 야간종가 - 1| 중앙값 = {gaps[align] * 100:.3f}%"
        )

    best = min(gaps, key=gaps.get)
    print(f"  → '{best}' 선택")

    return best, datasets[best]

In [ ]:
align, night = resolve_alignment(fut, trading_days)

if night.empty:
    raise RuntimeError("야간세션과 정규장을 매칭한 데이터가 없습니다.")

print("사용 월물:", ", ".join(night["contract_name"].unique()))
display(night.head())

### 분석용 패널

(날짜, 종목) 한 행에 야간선물 수익률과 종목 시가갭을 붙인다.

$$\text{gap}_{i,t} = \frac{P^{open}_{i,t}}{P^{base}_{i,t}} - 1$$

In [ ]:
def build_panel(stocks, night):
    """(날짜, 종목) 한 행:  night_ret[t],  gap[i,t]"""
    panel = stocks.merge(night[["night_ret"]], left_on="date", right_index=True)

    # gap[i,t] = P_open[i,t] / P_base[i,t] - 1
    panel["gap"] = panel["open"] / panel["base"] - 1

    return panel


panel = build_panel(stocks, night)

print(f"패널: {len(panel):,}행, {panel['date'].min():%Y-%m-%d} ~ {panel['date'].max():%Y-%m-%d}")
display(panel.head())

## 4. 분석 1 — 상관관계 (전체 기간)

종목 $i$마다 전체 기간(`DATA_START` ~ `BACKTEST_END`)으로 계산한다.

$$\rho_i = \operatorname{corr}\left(\text{night\_ret}_t,\ \text{gap}_{i,t}\right)
\qquad
95\%\ \text{CI} = \tanh\left(\operatorname{atanh}(\rho_i) \pm \frac{1.96}{\sqrt{n-3}}\right)$$

$$\text{gap}_{i,t} = a_i + b_i \cdot \text{night\_ret}_t + \varepsilon_{i,t}
\qquad (\text{OLS, 이분산 강건 표준오차 HC1})$$

- $b_i$ : 야간선물이 1% 움직일 때 종목 시가갭이 움직이는 % (**민감도**)
- $p < 0.05$ 이면 상관이 통계적으로 유의 → `상관유의 = O`

In [ ]:
def correlation_test(panel):
    rows = []

    for code, g in panel.groupby("code"):
        n = len(g)
        rho = g["night_ret"].corr(g["gap"])
        half_width = 1.96 / np.sqrt(n - 3)

        fit = smf.ols("gap ~ night_ret", data=g).fit(cov_type="HC1")

        rows.append({
            "code": code,
            "종목": STOCKS[code][0],
            "분야": STOCKS[code][1],
            "시장": g["market"].iloc[0],
            "n": n,
            "상관계수": rho,
            "CI 하한": np.tanh(np.arctanh(rho) - half_width),
            "CI 상한": np.tanh(np.arctanh(rho) + half_width),
            "a": fit.params["Intercept"],
            "b": fit.params["night_ret"],
            "b t값": fit.tvalues["night_ret"],
            "p값": fit.pvalues["night_ret"],
            "R²": fit.rsquared,
        })

    return pd.DataFrame(rows).set_index("code")


corr = correlation_test(panel)
display(corr.sort_values("상관계수", ascending=False).round(3))

## 5. 분석 2 — Walk-forward 예측 (백테스트 기간)

백테스트 기간의 거래일 $t$마다, 종목 $i$별로

1. **학습**: $t-1$부터 거꾸로 데이터가 있는 거래일 `TRAIN_WINDOW`(=60)개로 $\hat a_i, \hat b_i$ 추정
2. **예측**:
$$\widehat{\text{gap}}_{i,t} = \hat a_i + \hat b_i \cdot \text{night\_ret}_t
\qquad\Rightarrow\qquad
\hat P^{open}_{i,t} = P^{base}_{i,t}\left(1 + \widehat{\text{gap}}_{i,t}\right)$$
3. **비교 기준 (random walk)**: $\hat P^{open}_{i,t} = P^{base}_{i,t}$ (갭 0, 전일 종가 그대로)

### 오차 지표
종목마다 주가 수준이 달라 원 단위 대신 **갭 기준(%p)** 으로 비교한다.

| 지표 | 정의 |
|---|---|
| 오차 | $e_{i,t} = \widehat{\text{gap}}_{i,t} - \text{gap}_{i,t} \;=\; (\hat P^{open} - P^{open}) / P^{base}$ |
| MAE | $\operatorname{mean}\lvert e \rvert$ |
| 방향적중률 | $\operatorname{sign}(\widehat{\text{gap}}) = \operatorname{sign}(\text{gap})$ 인 날의 비율 (gap = 0 인 날 제외) |
| OOS R² | $1 - \sum e^2 \,/\, \sum \text{gap}^2$ — 0보다 크면 야간선물이 예측을 개선 → `예측개선 = O` |

In [ ]:
def walk_forward(panel):
    results = []

    for code, g in panel.groupby("code"):
        g = g.set_index("date").sort_index()

        for t in g.loc[BACKTEST_START:BACKTEST_END].index:
            train = g[g.index < t].tail(TRAIN_WINDOW)

            if len(train) < MIN_TRAIN:
                print(f"[건너뜀] {STOCKS[code][0]} {t:%Y-%m-%d}: 학습 표본 {len(train)}개 < {MIN_TRAIN}")
                continue

            today = g.loc[t]

            fit = smf.ols("gap ~ night_ret", data=train).fit()
            a, b = fit.params["Intercept"], fit.params["night_ret"]
            gap_hat = a + b * today["night_ret"]

            results.append({
                "date": t,
                "code": code,
                "name": today["name"],
                "train_from": train.index[0],
                "train_to": train.index[-1],
                "n_train": len(train),
                "a": a,
                "b": b,
                "night_ret": today["night_ret"],
                "gap": today["gap"],
                "gap_hat": gap_hat,
                "base": today["base"],
                "actual": today["open"],
                "pred_return": today["base"] * (1 + gap_hat),
                "pred_random_walk": today["base"],
            })

    if not results:
        raise RuntimeError("예측 가능한 테스트일이 없습니다.")

    res = pd.DataFrame(results)

    # 오차 (갭 기준) = (P̂_open - P_open) / P_base
    res["err_return"] = res["gap_hat"] - res["gap"]
    res["err_random_walk"] = 0 - res["gap"]

    return res


res = walk_forward(panel)

print(f"예측 {len(res):,}건 ({res['code'].nunique()}종목 × {res['date'].nunique()}거래일)")
display(res.head())

In [ ]:
def evaluate(res):
    rows = []

    for code, g in res.groupby("code"):
        moved = g["gap"] != 0

        rows.append({
            "code": code,
            "기간 n": len(g),
            "기간 상관계수": g["night_ret"].corr(g["gap"]),
            "MAE(%p)": g["err_return"].abs().mean() * 100,
            "MAE_RW(%p)": g["err_random_walk"].abs().mean() * 100,
            "방향적중률(%)": (
                np.sign(g["gap_hat"][moved]) == np.sign(g["gap"][moved])
            ).mean() * 100,
            "OOS R²": 1 - (g["err_return"] ** 2).sum() / (g["err_random_walk"] ** 2).sum(),
        })

    return pd.DataFrame(rows).set_index("code")


def summarize(corr, metrics):
    summary = corr.join(metrics)
    summary["상관유의"] = np.where(summary["p값"] < SIGNIFICANCE, "O", "X")
    summary["예측개선"] = np.where(summary["OOS R²"] > 0, "O", "X")
    return summary.sort_values("상관계수", ascending=False)


metrics = evaluate(res)
summary = summarize(corr, metrics)

print("===== 분석 1. 상관관계 (전체 기간) =====")
display(summary[[
    "종목", "분야", "시장", "n", "상관계수", "CI 하한", "CI 상한",
    "b", "b t값", "p값", "R²", "상관유의",
]].round(3))

print(f"===== 분석 2. Walk-forward 예측 ({BACKTEST_START} ~ {BACKTEST_END}) =====")
display(summary[[
    "종목", "기간 n", "기간 상관계수", "MAE(%p)", "MAE_RW(%p)",
    "방향적중률(%)", "OOS R²", "예측개선",
]].round(3))

n_sig = (summary["상관유의"] == "O").sum()
n_better = (summary["예측개선"] == "O").sum()
print(f"상관 유의 (p < {SIGNIFICANCE}): {n_sig}/{len(summary)}종목 | "
      f"예측 개선 (OOS R² > 0): {n_better}/{len(summary)}종목")

### 5-1. 오차 허용범위 분석

평균 오차는 하루의 큰 오차에 크게 끌려갈 수 있다 (예: 하나마이크론 8/18 하루가 제곱오차 합의 65%).
그래서 **엄격한 오차 기준** 안에 든 날이 20거래일 중 며칠인지 센다.

| 기준 | 의미 |
|---|---|
| $\lvert e_{i,t} \rvert \le 0.5\%p$ | 매우 엄격 (8월 평균 오차 약 1.5%p의 1/3) |
| $\lvert e_{i,t} \rvert \le 1.0\%p$ | 엄격 |

- **종목별 표**: 회귀와 기준(전일 종가 그대로) 각각 허용범위 안에 든 날 수, 최대 오차일, 그 하루가 제곱오차 합에서 차지하는 비중
- **날짜별 표**: 오차가 큰 날이 여러 종목에 동시에 몰리는지 확인. `실제가 더 높음`이 15에 가깝거나 0에 가까우면 업종 전체가 같은 방향으로 예측을 벗어난 날이다.
- `직전 휴장`: 직전 정규장과의 사이에 주말·연휴가 낀 날. 야간선물이 그사이 정보를 담지 못한다.

In [ ]:
TOLERANCES = [0.5, 1.0]   # 오차 허용범위 (%p)


def count_within_tolerance(res):
    """종목별로 |오차| ≤ 허용범위인 날의 수 (회귀 vs 전일 종가)와 최대 오차일의 영향"""
    rows = []

    for code, g in res.groupby("code"):
        reg = g["err_return"].abs() * 100        # |회귀 오차| (%p)
        rw = g["err_random_walk"].abs() * 100    # |전일 종가 오차| (%p)

        row = {"code": code, "종목": STOCKS[code][0], "거래일": len(g)}
        for tol in TOLERANCES:
            row[f"회귀 ≤{tol}%p"] = (reg <= tol).sum()
            row[f"기준 ≤{tol}%p"] = (rw <= tol).sum()

        worst = reg.idxmax()
        row["최대오차(%p)"] = reg[worst]
        row["최대오차일"] = g.at[worst, "date"].strftime("%m-%d")
        row["최대오차일 비중(%)"] = reg[worst] ** 2 / (reg ** 2).sum() * 100   # 제곱오차 합 중 하루의 비중
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .set_index("code")
        .sort_values([f"회귀 ≤{tol}%p" for tol in TOLERANCES], ascending=False)
    )


def errors_by_date(res, tol=TOLERANCES[-1]):
    """날짜별로 오차가 허용범위를 넘은 종목 수. 직전 정규장과의 사이에 휴장일이 낀 날은 표시한다."""
    reg = res.assign(
        err=res["err_return"] * 100,                 # 예측 - 실제 (%p), 음수면 실제가 예측보다 높게 시작
        abs_err=res["err_return"].abs() * 100,
    )
    g = reg.groupby("date")

    dates = g.size().index
    prev_trading_day = trading_days[trading_days.get_indexer(dates) - 1]
    days_since_prev = (dates - prev_trading_day).days

    table = pd.DataFrame({
        "야간선물(%)": g["night_ret"].first() * 100,
        "평균 |오차|(%p)": g["abs_err"].mean(),
        "평균 오차 방향(%p)": g["err"].mean(),
        "실제가 더 높음": g["err"].apply(lambda s: (s < 0).sum()),
        f"오차 >{tol}%p 종목수": g["abs_err"].apply(lambda s: (s > tol).sum()),
        "최대 오차 종목": g.apply(lambda d: d.loc[d["abs_err"].idxmax(), "name"], include_groups=False),
        "최대 오차(%p)": g["abs_err"].max(),
    })
    table["직전 휴장"] = np.where(days_since_prev > 3, "연휴 후",
                         np.where(days_since_prev > 1, "주말 후", ""))
    table.index = table.index.strftime("%m-%d")
    return table


within = count_within_tolerance(res)
by_date = errors_by_date(res)

print(f"===== 종목별: 20거래일 중 오차 허용범위 안에 든 날 =====")
display(within.round(2))

print(f"===== 날짜별: 오차 > {TOLERANCES[-1]}%p 종목 수 =====")
display(by_date.round(2))

## 6. 시각화

In [ ]:
COLOR_MAIN = "#2a78d6"
COLOR_MUTED = "#a3a29c"
TEXT_SECONDARY = "#52514e"
GRID = "#e4e3df"


def _set_korean_font():
    installed = {f.name for f in font_manager.fontManager.ttflist}

    for font in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
        if font in installed:
            plt.rcParams["font.family"] = font
            break

    plt.rcParams["axes.unicode_minus"] = False


def _style(ax, title, grid_axis="both"):
    ax.set_title(title, loc="left", fontsize=11)
    ax.grid(True, axis=grid_axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=TEXT_SECONDARY, labelsize=8)

    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(GRID)


_set_korean_font()

In [ ]:
def plot_summary(summary):
    """종목별 상관계수(95% CI)와 백테스트 기간 OOS R²"""
    data = summary.iloc[::-1].reset_index(drop=True)   # 위에서부터 상관계수 큰 순
    y = np.arange(len(data))
    significant = (data["p값"] < SIGNIFICANCE).to_numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 0.45 * len(data) + 2), sharey=True)

    # (1) 상관계수 + 95% 신뢰구간
    ax = axes[0]
    for mask, color, label in [
        (significant, COLOR_MAIN, f"p < {SIGNIFICANCE}"),
        (~significant, COLOR_MUTED, f"p ≥ {SIGNIFICANCE}"),
    ]:
        if not mask.any():
            continue

        rho = data["상관계수"].to_numpy()[mask]
        ci_low = data["CI 하한"].to_numpy()[mask]
        ci_high = data["CI 상한"].to_numpy()[mask]

        ax.barh(y[mask], rho, height=0.6, color=color, label=label)
        ax.errorbar(
            rho, y[mask],
            xerr=[rho - ci_low, ci_high - rho],
            fmt="none", ecolor=TEXT_SECONDARY, elinewidth=1, capsize=2,
        )

    ax.axvline(0, color=TEXT_SECONDARY, lw=1)
    ax.set_yticks(y, data["종목"])
    _style(ax, f"상관계수 ρ (night_ret vs gap, 95% CI)\n{DATA_START:%Y-%m-%d} ~ {BACKTEST_END}",
           grid_axis="x")
    ax.legend(fontsize=8, frameon=False, loc="lower right")

    # (2) OOS R²
    ax = axes[1]
    ax.barh(y, data["OOS R²"], height=0.6, color=COLOR_MAIN)
    ax.axvline(0, color=TEXT_SECONDARY, lw=1)

    for yi, value in zip(y, data["OOS R²"]):
        ax.annotate(f"{value:.2f}", (value, yi), xytext=(4 if value >= 0 else -4, 0),
                    textcoords="offset points", va="center",
                    ha="left" if value >= 0 else "right",
                    fontsize=8, color=TEXT_SECONDARY)

    _style(ax, f"OOS R² (전일 종가 대비 예측 개선, > 0 이면 개선)\n"
               f"{BACKTEST_START} ~ {BACKTEST_END} walk-forward", grid_axis="x")

    fig.suptitle("코스피200 야간선물 ↔ 반도체 종목 시가갭", fontsize=13, x=0.01, ha="left")
    fig.tight_layout()

    return fig


fig_summary = plot_summary(summary)
plt.show()

In [ ]:
def plot_scatter_grid(panel, summary):
    """종목별 산점도: night_ret(%) vs gap(%), 회귀선은 전체 기간 OLS"""
    codes = list(summary.index)
    n_cols = 5
    n_rows = int(np.ceil(len(codes) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3.0 * n_rows),
                             sharex=True, squeeze=False)

    in_backtest = panel["date"].between(BACKTEST_START, BACKTEST_END)

    for ax, code in zip(axes.flat, codes):
        g = panel[panel["code"] == code]
        bt = in_backtest[g.index]
        row = summary.loc[code]

        ax.scatter(g["night_ret"][~bt] * 100, g["gap"][~bt] * 100, s=14,
                   color=COLOR_MUTED, edgecolor="white", linewidth=0.8, label="학습 기간")
        ax.scatter(g["night_ret"][bt] * 100, g["gap"][bt] * 100, s=20,
                   color=COLOR_MAIN, edgecolor="white", linewidth=0.8, label="백테스트 기간")

        x_line = np.linspace(g["night_ret"].min(), g["night_ret"].max(), 20)
        ax.plot(x_line * 100, (row["a"] + row["b"] * x_line) * 100,
                color=TEXT_SECONDARY, lw=1.5)

        _style(ax, f"{row['종목']}  ρ={row['상관계수']:.2f}, b={row['b']:.2f}")
        ax.title.set_fontsize(9)

    for ax in axes.flat[len(codes):]:
        ax.set_visible(False)

    for ax in axes[-1]:
        ax.set_xlabel("night_ret (%)", color=TEXT_SECONDARY, fontsize=8)
    for ax in axes[:, 0]:
        ax.set_ylabel("gap (%)", color=TEXT_SECONDARY, fontsize=8)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", fontsize=8, frameon=False, ncol=2)
    fig.suptitle("야간선물 수익률 vs 종목 시가갭", fontsize=13, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.97))

    return fig


fig_scatter = plot_scatter_grid(panel, summary)
plt.show()

In [ ]:
def plot_daily_errors(res, summary):
    """백테스트 기간 일별 |오차| : 전 종목 평균, return 회귀 vs random_walk"""
    daily = res.groupby("date")[["err_return", "err_random_walk"]].apply(
        lambda g: g.abs().mean() * 100
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(daily.index, daily["err_random_walk"], color=COLOR_MUTED, lw=2,
            marker="o", ms=4, ls="--", label="random_walk (전일 종가)")
    ax.plot(daily.index, daily["err_return"], color=COLOR_MAIN, lw=2,
            marker="o", ms=5, label="return 회귀 (야간선물)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
    _style(ax, f"일별 평균 절대오차 (%p, {len(summary)}종목 평균)")
    ax.legend(fontsize=8, frameon=False)
    fig.tight_layout()

    return fig


fig_daily = plot_daily_errors(res, summary)
plt.show()

In [ ]:
def plot_error_heatmap(res, within, by_date):
    """종목 × 날짜 |오차|(%p). 색이 진할수록 오차가 크다."""
    table = (
        res.assign(abs_err=res["err_return"].abs() * 100)
        .pivot(index="code", columns="date", values="abs_err")
        .reindex(within.index)
    )
    values = table.to_numpy(dtype=float)
    vmax = 6

    cmap = LinearSegmentedColormap.from_list("sequential", ["#f4f7fb", COLOR_MAIN, "#123a6b"])

    fig, ax = plt.subplots(figsize=(14, 0.42 * len(table) + 2.2))
    image = ax.imshow(values, cmap=cmap, vmin=0, vmax=vmax, aspect="auto")

    for (r, c), v in np.ndenumerate(values):
        if np.isfinite(v):
            ax.text(c, r, f"{v:.1f}", ha="center", va="center", fontsize=7.5,
                    color="white" if v > vmax * 0.45 else TEXT_SECONDARY)

    holiday = by_date["직전 휴장"].reindex(table.columns.strftime("%m-%d")).to_numpy()
    labels = [f"{d:%m-%d}" + ("*" if h else "") for d, h in zip(table.columns, holiday)]

    ax.set_xticks(np.arange(len(table.columns)), labels, rotation=45)
    ax.set_yticks(np.arange(len(table)), within["종목"])
    ax.tick_params(colors=TEXT_SECONDARY, labelsize=8)
    for spine in ax.spines.values():
        spine.set_visible(False)

    colorbar = fig.colorbar(image, ax=ax, shrink=0.8, extend="max")
    colorbar.set_label("|예측 오차| (%p)", color=TEXT_SECONDARY, fontsize=9)

    ax.set_title(
        f"종목 × 날짜 예측 오차 (%p)   * = 주말·연휴 다음 날   "
        f"종목은 {TOLERANCES[0]}%p 안에 든 날이 많은 순",
        loc="left", fontsize=11,
    )
    fig.tight_layout()

    return fig


fig_heatmap = plot_error_heatmap(res, within, by_date)
plt.show()

## 7. 결과 저장

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tag = f"semiconductor_{pd.Timestamp(BACKTEST_START):%Y%m}"

summary.to_csv(OUTPUT_DIR / f"night_futures_{tag}_summary.csv", encoding="utf-8-sig")
res.to_csv(OUTPUT_DIR / f"night_futures_{tag}_predictions.csv", index=False, encoding="utf-8-sig")
within.to_csv(OUTPUT_DIR / f"night_futures_{tag}_within_tolerance.csv", encoding="utf-8-sig")
by_date.to_csv(OUTPUT_DIR / f"night_futures_{tag}_errors_by_date.csv", encoding="utf-8-sig")

for key, fig in {"summary": fig_summary, "scatter": fig_scatter, "daily_errors": fig_daily,
                 "error_heatmap": fig_heatmap}.items():
    fig.savefig(OUTPUT_DIR / f"night_futures_{tag}_{key}.png", dpi=150)

print("결과 저장:", OUTPUT_DIR)

## 결과 정리

| 항목 | 결과 |
|---|---|
| 상관 유의 종목 수 (p < 0.05) | (실행 후 작성) |
| 예측 개선 종목 수 (OOS R² > 0) | (실행 후 작성) |
| 민감도 b 가 큰 종목 | (실행 후 작성) |
| 상관이 약한 종목 | (실행 후 작성) |

### 해석 시 주의사항
- 삼성전자·SK하이닉스는 코스피200 비중이 커서, 두 종목의 상관이 높은 것은 어느 정도 당연한 결과다 (지수 자체가 두 종목을 많이 담고 있음).
- 8월 walk-forward 기간은 약 20거래일이라 OOS R²·방향적중률의 표본이 작다.
- KRX 일별 데이터는 원주가이므로 시가갭은 기준가(종가 − 전일대비)로 계산했다.